# RAG Legal Assistant Indonesia

Notebook ini memuat seluruh PDF pada folder `documents`, melakukan chunking eksplisit, metadata enrichment, vector database lokal FAISS, BM25, ensemble retriever, parent-child retrieval, HyDE minimal dua jawaban hipotetis, reranker CrossEncoder, fallback DuckDuckGo, sitasi, dan interface Gradio sederhana.


In [1]:
!pip install -q "langchain>=0.2.11" "langchain-community>=0.2.10" "langchain-huggingface>=0.0.3" "pypdf>=4.3.1" "faiss-cpu>=1.8.0" "rank_bm25>=0.2.2" "sentence-transformers>=3.0.1" "duckduckgo-search>=6.2.6" "gradio>=4.39.0" "unsloth[colab-new]" "transformers>=4.43.0" "accelerate" "bitsandbytes" "python-dotenv>=1.0.1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 721.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423

In [5]:
import os
from pathlib import Path
import torch
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

try:
    from langchain.retrievers import EnsembleRetriever
except ModuleNotFoundError:
    from langchain_classic.retrievers import EnsembleRetriever

from sentence_transformers import CrossEncoder
from duckduckgo_search import DDGS

load_dotenv()

DOCUMENT_DIR = Path("documents")
PDF_FILES = sorted(DOCUMENT_DIR.glob("*.pdf"))
CHILD_CHUNK_SIZE = 900
CHILD_CHUNK_OVERLAP = 180
PARENT_CHUNK_SIZE = 2600
PARENT_CHUNK_OVERLAP = 350
RETRIEVE_K = 5
RERANK_TOP_K = 3
RERANK_THRESHOLD = 0.18
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
FT_MODEL_REPO = os.getenv("FT_MODEL_REPO", "artapamudaid/legal-assistant-indonesia-qwen25-qlora-merged-16bit")
print(PDF_FILES)


[PosixPath('documents/PP Nomor 35 Tahun 2021.pdf'), PosixPath('documents/PP Nomor 5 Tahun 2021.pdf'), PosixPath('documents/PP Nomor 51 Tahun 2023.pdf'), PosixPath('documents/UU Nomor 6 Tahun 2023.pdf')]


In [6]:
def infer_doc_type(filename):
    name = filename.lower()
    if name.startswith("uu"):
        return "Undang-Undang"
    if name.startswith("pp"):
        return "Peraturan Pemerintah"
    return "Dokumen Hukum"

def enrich_metadata(doc):
    source = Path(doc.metadata.get("source", "unknown")).name
    doc.metadata.update({
        "source_file": source,
        "doc_type": infer_doc_type(source),
        "page": int(doc.metadata.get("page", 0)) + 1,
        "jurisdiction": "Indonesia",
        "domain": "ketenagakerjaan-perizinan-kepatuhan",
    })
    return doc

pages = []
for pdf in PDF_FILES:
    loader = PyPDFLoader(str(pdf))
    pages.extend(enrich_metadata(doc) for doc in loader.load())
print("Total halaman:", len(pages))
print(pages[0].metadata)


Total halaman: 1949
{'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'documents/PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 1, 'page_label': '1', 'source_file': 'PP Nomor 35 Tahun 2021.pdf', 'doc_type': 'Peraturan Pemerintah', 'jurisdiction': 'Indonesia', 'domain': 'ketenagakerjaan-perizinan-kepatuhan'}


In [7]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=CHILD_CHUNK_SIZE, chunk_overlap=CHILD_CHUNK_OVERLAP)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=PARENT_CHUNK_SIZE, chunk_overlap=PARENT_CHUNK_OVERLAP)

parent_docs = parent_splitter.split_documents(pages)
child_docs = []
for parent_id, parent in enumerate(parent_docs):
    chunks = child_splitter.split_documents([parent])
    for child_id, child in enumerate(chunks):
        child.metadata.update(parent.metadata)
        child.metadata["parent_id"] = parent_id
        child.metadata["child_id"] = child_id
    child_docs.extend(chunks)

parent_lookup = {i: doc for i, doc in enumerate(parent_docs)}
print("Parent chunks:", len(parent_docs))
print("Child chunks:", len(child_docs))


Parent chunks: 1955
Child chunks: 3421


In [8]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
vectorstore = FAISS.from_documents(child_docs, embeddings)
vectorstore.save_local("faiss_legal_index")
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVE_K})

bm25_retriever = BM25Retriever.from_documents(child_docs)
bm25_retriever.k = RETRIEVE_K
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=[0.35, 0.65],
)
print("FAISS index saved to faiss_legal_index")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index saved to faiss_legal_index


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

USE_GPU = torch.cuda.is_available()
print("GPU tersedia:", USE_GPU)

if USE_GPU:
    # Import Unsloth hanya ketika GPU tersedia. Unsloth akan error jika runtime CPU.
    import unsloth
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=FT_MODEL_REPO,
        max_seq_length=4096,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
else:
    # Fallback CPU agar pipeline RAG tetap bisa diuji saat kuota GPU Colab habis.
    # Untuk penilaian final, jalankan lagi dengan GPU agar memakai Unsloth + model fine-tuned secara optimal.
    tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_REPO, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        FT_MODEL_REPO,
        device_map="cpu",
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.eval()

def generate_text(prompt, max_new_tokens=512, temperature=0.2):
    messages = [
        {"role": "system", "content": "Anda adalah asisten AI legal internal. Jawab hanya berdasarkan konteks hukum yang diberikan. Jika konteks tidak cukup, katakan tidak cukup informasi."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    device = "cuda" if USE_GPU else "cpu"
    inputs = inputs.to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()


GPU tersedia: False


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.03k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [10]:
def hyde_queries(question):
    prompt = f"""Buat 2 jawaban hipotetis singkat dalam bahasa Indonesia yang mungkin relevan untuk query hukum berikut.
Query: {question}
Format:
1. ...
2. ..."""
    hyde = generate_text(prompt, max_new_tokens=256, temperature=0.7)
    return [question, hyde]

def retrieve_with_parent_child(question, metadata_filter=None):
    queries = hyde_queries(question)
    candidates = []
    seen = set()
    for query in queries:
        for doc in ensemble_retriever.invoke(query):
            if metadata_filter:
                if any(doc.metadata.get(k) != v for k, v in metadata_filter.items()):
                    continue
            key = (doc.metadata.get("source_file"), doc.metadata.get("page"), doc.metadata.get("parent_id"), doc.page_content[:80])
            if key not in seen:
                seen.add(key)
                candidates.append(doc)
    parent_ids = []
    for doc in candidates:
        pid = doc.metadata.get("parent_id")
        if pid is not None and pid not in parent_ids:
            parent_ids.append(pid)
    return [parent_lookup[pid] for pid in parent_ids[:10]]


In [13]:
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

def rerank(question, docs):
    if not docs:
        return [], 0.0

    pairs = [[question, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda item: float(item[1]), reverse=True)
    top_score = float(ranked[0][1])

    return ranked[:RERANK_TOP_K], top_score

def web_search(question, max_results=3):
    results = DDGS().text(question + " regulasi Indonesia", max_results=max_results)

    return "\n".join(
        f"[{i+1}] {r.get('title')} - {r.get('href')}\n{r.get('body')}"
        for i, r in enumerate(results)
    )

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [1]:
def generate_text(prompt, max_new_tokens=512, temperature=0.2):
    messages = [
        {
            "role": "system",
            "content": "Anda adalah asisten AI legal internal. Jawab hanya berdasarkan konteks hukum yang diberikan. Jika konteks tidak cukup, katakan tidak cukup informasi."
        },
        {"role": "user", "content": prompt},
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    device = "cuda" if USE_GPU else "cpu"

    if hasattr(encoded, "to"):
        encoded = encoded.to(device)

    if isinstance(encoded, dict) or hasattr(encoded, "data"):
        input_ids = encoded["input_ids"]
        attention_mask = encoded.get("attention_mask", None)

        generate_kwargs = {"input_ids": input_ids}
        if attention_mask is not None:
            generate_kwargs["attention_mask"] = attention_mask
    else:
        input_ids = encoded.to(device)
        generate_kwargs = {"input_ids": input_ids}

    with torch.no_grad():
        outputs = model.generate(
            **generate_kwargs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = input_ids.shape[-1]

    return tokenizer.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True,
    ).strip()

In [ ]:
def format_sources(ranked_docs):
    sources = []
    for i, (doc, score) in enumerate(ranked_docs, start=1):
        meta = doc.metadata
        sources.append(
            f"[{i}] {meta.get('source_file', 'Dokumen')} - "
            f"hal. {meta.get('page', '-')}, skor rerank {float(score):.4f}"
        )
    return "\n".join(sources)


def build_context(ranked_docs):
    context_blocks = []
    for i, (doc, score) in enumerate(ranked_docs, start=1):
        meta = doc.metadata
        context_blocks.append(
            f"[Sumber {i}: {meta.get('source_file', 'Dokumen')}, "
            f"halaman {meta.get('page', '-')}, skor {float(score):.4f}]\n"
            f"{doc.page_content}"
        )
    return "\n\n---\n\n".join(context_blocks)


def answer_legal_question(question, metadata_filter=None, use_web_fallback=True):
    question = str(question).strip()
    if not question:
        return "Silakan masukkan pertanyaan legal terlebih dahulu."

    retrieved_docs = retrieve_with_parent_child(question, metadata_filter=metadata_filter)
    ranked_docs, top_score = rerank(question, retrieved_docs)

    web_context = ""
    if (not ranked_docs or top_score < RERANK_THRESHOLD) and use_web_fallback:
        web_context = web_search(question)

    legal_context = build_context(ranked_docs)
    source_list = format_sources(ranked_docs)

    if not legal_context and not web_context:
        return "Tidak cukup informasi dalam dokumen maupun pencarian web untuk menjawab pertanyaan tersebut."

    prompt = f"""Jawab pertanyaan legal berikut dalam bahasa Indonesia.
Gunakan konteks dokumen internal sebagai dasar utama. Jika ada konteks web, gunakan hanya sebagai pendukung dan sebutkan keterbatasannya.
Berikan jawaban ringkas, praktis, dan sertakan sitasi sumber internal dengan format [Sumber 1], [Sumber 2] jika relevan.
Jika konteks tidak cukup untuk menyimpulkan, nyatakan bahwa informasinya belum cukup.

Pertanyaan:
{question}

Konteks dokumen internal:
{legal_context if legal_context else 'Tidak ada konteks internal yang cukup relevan.'}

Konteks web fallback:
{web_context if web_context else 'Tidak digunakan.'}

Daftar sumber internal:
{source_list if source_list else 'Tidak ada sumber internal.'}
"""

    answer = generate_text(prompt, max_new_tokens=700, temperature=0.2)

    if source_list:
        return f"{answer}\n\nSumber internal:\n{source_list}"
    return answer


In [2]:
import gradio as gr

def gradio_answer(question):
    return answer_legal_question(question)

demo = gr.Interface(
    fn=gradio_answer,
    inputs=gr.Textbox(label="Pertanyaan Legal", lines=3),
    outputs=gr.Textbox(label="Jawaban AI Legal", lines=14),
    title="Asisten AI Legal Internal Indonesia",
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://21a440db4ca12a6e04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://21a440db4ca12a6e04.gradio.live


In [23]:
# Alternatif interface loop Python.
while False:
    question = input("Pertanyaan legal: ")
    if question.lower().strip() in {"exit", "quit", "keluar"}:
        break
    display(Markdown(answer_legal_question(question)))


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag